In [1]:
import time
import numpy as np


conv1_w=np.load('LeNet/LeNet-weights/Conv1_weight.np.npy')
conv1_b=np.load('LeNet/LeNet-weights/Conv1_bias.np.npy')
conv2_w=np.load('LeNet/LeNet-weights/Conv2_weight.np.npy')
conv2_b=np.load('LeNet/LeNet-weights/Conv2_bias.np.npy')
f1_w=np.load('LeNet/LeNet-weights/fc1_weight.np.npy')
f2_w=np.load('LeNet/LeNet-weights/fc2_weight.np.npy')
f1_b=np.load('LeNet/LeNet-weights/fc1_bias.np.npy')
f2_b=np.load('LeNet/LeNet-weights/fc2_bias.np.npy')
data=np.load('Mnist/mnist_data.npy')
target1=np.load('Mnist/mnist_label.npy')


In [2]:
from LeNet import Accelerator
from LeNet import Accelerator_Copy1
conv1=Accelerator.Convolution2D(1,10,5,5,1,1,0,0,conv1_w,conv1_b,1000000)
conv2=Accelerator.Convolution2D(10,20,5,5,1,1,0,0,conv2_w,conv2_b,1000000)
pool1=Accelerator.Pool(2,2,2,2,'Max',0,1,10000)
pool2=Accelerator_Copy1.Pool(2,2,2,2,'Max',0,1,10000) # duplicate layer for pipelining
fc1=Accelerator_Copy1.FC(320,50,1,f1_w,f1_b)
fc2=Accelerator_Copy1.FC(50,10,1,f2_w,f2_b)

/usr/local/lib/python3.6/dist-packages/pynq/pl_server/device.py:594: UserWarning: Users will not get PARAMETERS / REGISTERS information through TCL files. HWH file is recommended.
  warnings.warn(message, UserWarning)
/usr/local/lib/python3.6/dist-packages/pynq/lib/dma.py:200: UserWarning: Failed to find parameter c_sg_length_width; users should really use *.hwh files for overlays.
  warnings.warn(message, UserWarning)


In [3]:
# This is the pipelined in software version of the accelerator
import threading
import time
import numpy as np

Test_number = 10

# create thread class for a layer, where input and output queues are locked, 
# and additions to the output queue will wake up threads waiting for inputs
class LayerThread(threading.Thread):
    def __init__(self, thread_num, stop_thread):
        threading.Thread.__init__(self)
        self.thread_num = thread_num
        self.stop_thread = False
        self.input_lock = input_lock
        self.output_lock = output_lock

     def run(self):
        while not self.stop_thread:
                
            with self.input_lock:
                while not self.input_queue and not self.stop_thread:
                    self.input_condition.wait()
                if self.stop_thread:
                    break
                
                data = self.input_queue.pop(0)
                #print(f"layer {self.layer_name} data {data}")
            # add try except to stop all threads
            try:
                if self.thread_num == 0
                    x = conv1.forward(data2[i0],Accelerator.dma)
                    x = pool1.forward(x,Accelerator.dma)
                    result = conv2.forward(x,Accelerator.dma)
                else
                    x= pool1.forward(x,Accelerator.dma)
                    x=fc1.forward(x,Accelerator.dma)
                    rs[i0]=fc2.forward(x,Accelerator.dma)
            except Exception as e:
                print(e)
                print(f"layer {self.layer_name} encountered error")
            
            with self.output_lock:
                if type(self.output_queue) is np.ndarray:
                    self.output_queue[self.output_counter] = result
                    self.output_counter +=1
                else:
                    self.output_queue.append(result)
                self.output_condition.notify_all()

def test(testnumber):
    total = 0
    correct = 0
    data2 = data[0:testnumber]
    target=target1[0:testnumber]
    size = data2.shape
    t1=time.time()
    rs = np.zeros((size[0],10))
    
    # initialize buffers for pipelining
    conv1_buffer = []
    conv2_buffer = []
    pool1_buffer = []
    pool2_buffer = []
    fc1_buffer = []
    fc2_buffer = []
    
    thread0_lock = threading.Lock()
    thread1_lock = threading.Lock()
    
    thread1_lock = threading.Condition(conv1_lock)
    pool1_condition = threading.Condition(pool1_lock)
    
    for i0 in range(size[0]):
        if i0 < size[0]:
            conv1_buffer.append(conv1.forward(data2[i0],Accelerator.dma))
            
        if len(conv1_buffer) > 0:
            pool1_buffer.append(pool1.forward(conv1_buffer.pop(0),Accelerator.dma))
            
        if len(pool1_buffer) > 0:
            conv2_buffer.append(conv2.forward(pool1_buffer.pop(0),Accelerator.dma))
            
        if len(conv2_buffer) > 0:
            pool2_buffer.append(pool2.forward(conv2_buffer.pop(0),Accelerator_Copy1.dma))
        
        if len(pool2_buffer) > 0:
            fc1_buffer.append(fc1.forward(pool2_buffer.pop(0),Accelerator_Copy1.dma))
            
        if len(fc1_buffer) > 0:
            rs[i0]=fc2.forward(fc1_buffer.pop(0),Accelerator_Copy1.dma)
    for i in range(size[0]):
        if np.argmax(rs[i]) == target[i]:
            correct += 1
        total += 1
        
    t2=time.time()
    print('Inference Time',t2-t1)    
    print ('accuracy=',float(correct)/float(total))


In [4]:
Test_number=1  # number of images  for testing procedure
test(1)
test(5)
test(10)
test(50)
test(100)
test(200)
test(1000)

Inference Time 0.1830294132232666
accuracy= 1.0
Inference Time 0.8181633949279785
accuracy= 1.0
Inference Time 1.7313940525054932
accuracy= 1.0
Inference Time 8.101707458496094
accuracy= 0.98
Inference Time 16.30334711074829
accuracy= 0.99
Inference Time 32.57757306098938
accuracy= 0.975
Inference Time 164.3443922996521
accuracy= 0.979


In [11]:
# This is the non pipelined version of the accelerator
def test(testnumber):
    total = 0
    correct = 0
    data2 = data[0:testnumber]
    target=target1[0:testnumber]
    size = data2.shape
    t1=time.time()
    rs = np.zeros((size[0],10))
    for i0 in range(0, size[0]):
        x=conv1.forward(data2[i0],Accelerator.dma)
        x= pool1.forward(x,Accelerator.dma)
        x=conv2.forward(x,Accelerator.dma)
        x= pool1.forward(x,Accelerator.dma)
        x=fc1.forward(x,Accelerator.dma)
        rs[i0]=fc2.forward(x,Accelerator.dma)

    for i in range(0, size[0]):
        if np.argmax(rs[i])==target[i]:
            #print(np.argmax(rs[i]),target[i])
            correct+=1
        total+=1
    t2=time.time()
    print('Inference Time',t2-t1)    
    print ('accuracy=',float(correct)/float(total))

Test_number=10  # number of images  for testing procedure
test(1)
"""test(1)
test(1)
test(1)
test(1)
test(1)
test(1)
test(1)
test(1)
test(1)
print()

test(5)
test(5)
test(5)
test(5)
test(5)
test(5)
test(5)
test(5)
test(5)
test(5)
print()

test(10)
test(10)
test(10)
test(10)
test(10)
test(10)
test(10)
test(10)
test(10)
test(10)
print()

test(50)
test(50)
test(50)
test(50)
test(50)
test(50)
test(50)
test(50)
test(50)
test(50)
print()

test(100)
test(100)
test(100)
test(100)
test(100)
test(100)
test(100)
test(100)
test(100)
test(100)
print()

test(200)
test(200)
test(200)
test(200)
test(200)
test(200)
test(200)
test(200)
test(200)
test(200)
print()


test(1000)
test(1000)
test(1000)
test(1000)
test(1000)
test(1000)
test(1000)
test(1000)
test(1000)
test(1000)
print()"""


ValueError: cannot reshape array of size 0 into shape (10,12,2)

In [1]:
from LeNet import Accelerator1
import time
import numpy as np
# This is the asyncio pipelined version
import asyncio
from concurrent.futures import ThreadPoolExecutor

conv1_w=np.load('LeNet/LeNet-weights/Conv1_weight.np.npy')
conv1_b=np.load('LeNet/LeNet-weights/Conv1_bias.np.npy')
conv2_w=np.load('LeNet/LeNet-weights/Conv2_weight.np.npy')
conv2_b=np.load('LeNet/LeNet-weights/Conv2_bias.np.npy')
f1_w=np.load('LeNet/LeNet-weights/fc1_weight.np.npy')
f2_w=np.load('LeNet/LeNet-weights/fc2_weight.np.npy')
f1_b=np.load('LeNet/LeNet-weights/fc1_bias.np.npy')
f2_b=np.load('LeNet/LeNet-weights/fc2_bias.np.npy')
data=np.load('Mnist/mnist_data.npy')
target1=np.load('Mnist/mnist_label.npy')

executor1 = ThreadPoolExecutor()
conv1=Accelerator1.Convolution2D(1,10,5,5,1,1,0,0,conv1_w,conv1_b,1000000, executor=executor1)
conv2=Accelerator1.Convolution2D(10,20,5,5,1,1,0,0,conv2_w,conv2_b,1000000, executor=executor1)
pool1=Accelerator1.Pool(2,2,2,2,'Max',0,1,10000, executor=executor1)
pool2=Accelerator1.Pool(2,2,2,2,'Max',0,1,10000, executor=executor1) # duplicate layer for pipelining
fc1=Accelerator1.FC(320,50,1,f1_w,f1_b, executor=executor1)
fc2=Accelerator1.FC(50,10,1,f2_w,f2_b, executor=executor1)

/usr/local/lib/python3.6/dist-packages/pynq/pl_server/device.py:594: UserWarning: Users will not get PARAMETERS / REGISTERS information through TCL files. HWH file is recommended.
  warnings.warn(message, UserWarning)
/usr/local/lib/python3.6/dist-packages/pynq/lib/dma.py:200: UserWarning: Failed to find parameter c_sg_length_width; users should really use *.hwh files for overlays.
  warnings.warn(message, UserWarning)


In [3]:




async def process_layer(layer, input_queue, output_queue, dma, layer_name):
    while True:
        #data = await input_queue.get()
        try:
            data = await asyncio.wait_for(input_queue.get(), timeout=5.0)
            #print(f"{layer_name} data: {data}")
        except asyncio.TimeoutError:
            print(f"{layer_name} timed out waiting for the data")
            break
        if data is None:
            print(f"{layer_name} received None, exiting")
            break
            

        print(f"{layer_name} received data")
        #result = await layer.forward(data, dma)
        #result = await layer.forward(data,dma)
        result = await asyncio.get_event_loop().run_in_executor(executor1, layer.forward, data, dma)
        print(f"{layer_name} result: {result}")
        print(f"{layer_name} processed data")
        await output_queue.put(result)
        print(f"{layer_name} put data to next queue")
        
async def test(testnumber):
    total = 0
    correct = 0
    data2 = data[0:testnumber]
    target=target1[0:testnumber]
    size = data2.shape
    t1=time.time()
    results = np.zeros((size[0],10))
    
    conv1_queue = asyncio.Queue()
    conv2_queue = asyncio.Queue()
    pool1_queue = asyncio.Queue()
    pool2_queue = asyncio.Queue()
    fc1_queue = asyncio.Queue()
    fc2_queue = asyncio.Queue()
    
    tasks = [
        asyncio.ensure_future(process_layer(conv1 ,conv1_queue ,pool1_queue ,Accelerator1.dma, "conv1")),
        asyncio.ensure_future(process_layer(pool1 ,pool1_queue ,conv2_queue ,Accelerator1.dma, "pool1")),
        asyncio.ensure_future(process_layer(conv2 ,conv2_queue ,pool2_queue ,Accelerator1.dma, "conv2")),
        asyncio.ensure_future(process_layer(pool2 ,pool2_queue ,fc1_queue ,Accelerator1.dma, "pool2")),
        asyncio.ensure_future(process_layer(fc1 ,fc1_queue ,fc2_queue ,Accelerator1.dma, "fc1")),
        asyncio.ensure_future(process_layer(fc2 ,fc2_queue ,asyncio.Queue() ,Accelerator1.dma, "fc2"))
    ]
    
    # begin processing
    for i in range(size[0]):
        print(f"Putting data {i} into conv1_queue")
        await conv1_queue.put(data2[i])
        
    # collect results
    for i in range(size[0]):
        #results[i] = await fc2_queue.get()
        
        try:
            results[i] = await asyncio.wait_for(fc2_queue.get(), timeout=5.0)
            print(f"Collected result {i} from fc2_queue")
        except asyncio.TimeoutError:
            print(f"Timeout waiting for result {i} from fc2_queue")
            break
 
        
    # stop layer tasks
    for queue in [conv1_queue, pool1_queue, conv2_queue, pool2_queue, fc1_queue, fc2_queue]:
        await queue.put(None)
    await asyncio.gather(*tasks)
    
    for i in range(size[0]):
        if np.argmax(results[i]) == target[i]:
            correct += 1
        total += 1
    t2 = time.time()
    print('Inference Time',t2-t1)    
    print ('accuracy=',float(correct)/float(total))

Test_number=10  # number of images  for testing procedure
loop = asyncio.get_event_loop()
loop.run_until_complete(test(Test_number))

/usr/lib/python3/dist-packages/ipykernel_launcher.py:22: RuntimeWarning: coroutine 'FC.forward' was never awaited
/usr/lib/python3/dist-packages/ipykernel_launcher.py:22: RuntimeWarning: coroutine 'Pool.forward' was never awaited
/usr/lib/python3/dist-packages/ipykernel_launcher.py:22: RuntimeWarning: coroutine 'Convolution2D.forward' was never awaited


fc1 processed data
fc1 put data to next queue
pool2 processed data
pool2 put data to next queue
conv2 processed data
conv2 put data to next queue
pool1 processed data
pool1 put data to next queue
Putting data 0 into conv1_queue
Putting data 1 into conv1_queue
Putting data 2 into conv1_queue
Putting data 3 into conv1_queue
Putting data 4 into conv1_queue
Putting data 5 into conv1_queue
Putting data 6 into conv1_queue
Putting data 7 into conv1_queue
Putting data 8 into conv1_queue
Putting data 9 into conv1_queue
conv1 processed data
conv1 put data to next queue
fc2 received data
fc1 received data
pool2 received data
conv2 received data
pool1 received data
conv1 received data
conv1 received data
fc2 processed data
fc2 put data to next queue
fc1 processed data
fc1 put data to next queue
pool2 processed data
pool2 put data to next queue
conv2 processed data
conv2 put data to next queue
pool1 processed data
pool1 put data to next queue
conv1 processed data
conv1 put data to next queue
conv1 

TypeError: float() argument must be a string or a number, not 'coroutine'